# MODEL

In this file, readind the output data from feature_engineering.ipynb, we will:

1. Define the PanelSplit
2. Define the model
3. Apply cross_val_fit_predict to do the prediction 
4. Evaluation

In [41]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from panelsplit.cross_validation import PanelSplit
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, f1_score, roc_auc_score, 
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier  
import os
import itertools

First we load the data:

In [42]:
df = pd.read_parquet("../data_clean/final_data.parquet")

### DEFINE THE TARGET VARIABLE AND THE FEATURES

In [43]:
def prepare_ml_experiment(df, features_to_use=None, target_col='target_2m', keep_target_nans=True):
    df_temp = df.copy()
    
    # 1. HIDE THE FUTURE: Borramos directamente los últimos 2 meses absolutos (el futuro desconocido)
    last_2_months_mask = df_temp.groupby('iso3').cumcount(ascending=False) < 2
    df_temp = df_temp[~last_2_months_mask] 

    # 2. FEATURE SELECTION
    blacklist = [target_col, 'allocation_elegible', 'notes_acled', 'hdx_alert_level']
    features_cols = [col for col in features_to_use if col not in blacklist]

    # 3. CLEANING: Si keep_target_nans es True, NO borramos los NaNs de la variable objetivo
    if keep_target_nans:
        df_model = df_temp.dropna(subset=features_cols)
    else:
        df_model = df_temp.dropna(subset=features_cols + [target_col])
    
    X = df_model[features_cols]
    y = df_model[target_col]
    
    return X, y

Now we are going to define different groups of features and check with which combination the model performs better:

In [ ]:
# Only raw variables from IDMC and ACLED
features_baseline = ['monthly_displacement','fatalities','event_count','Battles','Explosions/Remote violence','Protests', 'Riots',
'Strategic developments','Violence against civilians', 'allocation_elegible']

# Features derived from IDMC (lags, rolling means, etc.) and from ACLED (lags, rolling means and also the "displacement score" that we created)
features_derived_humanitarian_impact = ['rolling_3m_displacements', 'disp_6m_avg','disp_3m_avg','monthly_displacement_lag1','monthly_displacement_lag2','disp_diff_3m_6m',
 'disp_change_lag1_to_now','disp_change_lag2_to_now','fat_6m_avg','fat_3m_avg','fatalities_lag1','fatalities_lag2','fat_diff_3m_6m',
 'fat_change_lag1_to_now','fat_change_lag2_to_now','bat_gr90','exp_gr150','bat_gr90_and_exp_gr150','bat_gr90_and_exp_gr150_last_5m',
 'bat_civ','conflict_severity_index','acled_disp_score_max','acled_disp_score_mean','acled_semantic_intensity',
 'acled_disp_events_count','acled_disp_events_ratio','lethality_rate','disp_per_fatality','disp_per_event']

# Features from EconAI data and HDX Signals
features_risk_alerts = ['risk_3', 'risk_12', 'logfat_risk_3', 'logfat_risk_12', 'hdx_value', 'hdx_alert_High concern', 'hdx_alert_Medium concern','risk3_6m_avg',
 'risk3_3m_avg','risk3_lag1','risk3_lag2','risk12_6m_avg','risk12_3m_avg','risk12_lag1','risk12_lag2','logfat_risk3_6m_avg',
 'logfat_risk3_3m_avg','logfat_risk3_lag1','logfat_risk3_lag2','logfat_risk12_6m_avg','logfat_risk12_3m_avg','logfat_risk12_lag1',
 'logfat_risk12_lag2','risk_gt_025','logfat_risk_3_gt_4','logfat_risk_3_high_last_5m','logfat_risk_12_above_5','logfat_risk_12_high_last_5m',
 'risk3_change_2m','risk12_change_2m','logfat_risk3_change_2m','logfat_risk12_change_2m','hdx_med_high_count','hdx_3m_sum',
 'hdx_medium_3m_sum','hdx_high_3m_sum','mean_value_hdx']

# Only features from INFORM Index
features_inform = ['INFORM', 'VU', 'CC', 'HA','INFORM_above_06','INFORM_high_last_5m']

# Only the signals that CERF has identified
features_cerf_signals = ['p_sig1', 'p_sig2', 'p_sig3', 'protracted_signal', 'h_sig1', 'h_sig2', 'hard_onset_signal']

We only want to keep the most relevant features: REGARDING THE F1-SCORE

In [45]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

def get_valuable_features_f1_pure(df, feature_list, target_col='target_2m', top_n=15):
    if len(feature_list) <= 3:
        return feature_list
        
    # 1. Preparamos datos MANTENIENDO los NaNs del target (necesario para aplicar tu regla)
    X, y = prepare_ml_experiment(df, features_to_use=feature_list, target_col=target_col, keep_target_nans=True)
    
    time_axis = pd.to_datetime(X.index.get_level_values('month'))
    max_date = time_axis.max()
    cutoff_date = max_date - pd.DateOffset(years=2)
    
    # 2. Separamos Train (para aprender) y Test (para evaluar qué features son buenas)
    train_mask_time = time_axis < cutoff_date
    test_mask_time = time_axis >= cutoff_date
    
    X_train_raw = X[train_mask_time]
    y_train_raw = y[train_mask_time]
    
    X_test_raw = X[test_mask_time]
    y_test_raw = y[test_mask_time]
    
    # --- APLICAMOS TU REGLA ---
    # TRAIN: Quitamos los NaNs para que el modelo aprenda onsets puros
    valid_train = y_train_raw.notna()
    X_train_clean = X_train_raw[valid_train]
    y_train_clean = y_train_raw[valid_train]
    
    # TEST: Rellenamos los NaNs (crisis) con 0 para penalizar falsas alarmas
    X_test_clean = X_test_raw
    y_test_clean = y_test_raw.fillna(0)
    # --------------------------
    
    # 3. Entrenamos el modelo con los datos puros
    model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
    model.fit(X_train_clean, y_train_clean)
    
    # 4. Evaluamos la importancia de las features en el TEST (Mundo real)
    # Al hacerlo en el test_clean (con 0s), las variables que provoquen Falsos Positivos serán penalizadas
    result = permutation_importance(
        model, X_test_clean, y_test_clean, 
        scoring='f1',       
        n_repeats=5, 
        random_state=42, 
        n_jobs=-1
    )
    
    # 5. Extraemos y seleccionamos las mejores
    importances = pd.Series(result.importances_mean, index=X.columns)
    best_features = importances.sort_values(ascending=False).head(top_n).index.tolist()
    
    return best_features

In [46]:
best_features_derived_humanitarian_impact = get_valuable_features_f1_pure(df, features_derived_humanitarian_impact)
best_features_risk_alerts = get_valuable_features_f1_pure(df, features_risk_alerts)

In [47]:
print("Best features from Derived Humanitarian Impact:", best_features_derived_humanitarian_impact)
print("Best features from Risk Alerts:", best_features_risk_alerts)

Best features from Derived Humanitarian Impact: ['disp_change_lag2_to_now', 'rolling_3m_displacements', 'disp_3m_avg', 'fat_3m_avg', 'fatalities_lag1', 'fatalities_lag2', 'disp_6m_avg', 'fat_6m_avg', 'lethality_rate', 'disp_change_lag1_to_now', 'disp_per_fatality', 'disp_per_event', 'conflict_severity_index', 'acled_semantic_intensity', 'acled_disp_score_max']
Best features from Risk Alerts: ['risk_3', 'logfat_risk12_6m_avg', 'logfat_risk12_lag1', 'logfat_risk12_lag2', 'risk_gt_025', 'logfat_risk_3_gt_4', 'logfat_risk_3_high_last_5m', 'logfat_risk_12_above_5', 'logfat_risk_12_high_last_5m', 'risk3_change_2m', 'risk12_change_2m', 'logfat_risk3_change_2m', 'logfat_risk12_change_2m', 'hdx_med_high_count', 'hdx_3m_sum']


In [48]:
best_general = get_valuable_features_f1_pure(df, features_baseline + features_derived_humanitarian_impact + features_risk_alerts + features_inform + features_cerf_signals, top_n=14)
print("Best features general:", best_general)

Best features general: ['monthly_displacement', 'logfat_risk12_3m_avg', 'logfat_risk_12', 'logfat_risk12_lag1', 'disp_3m_avg', 'rolling_3m_displacements', 'disp_6m_avg', 'logfat_risk3_3m_avg', 'logfat_risk12_6m_avg', 'risk_12', 'logfat_risk3_lag1', 'VU', 'INFORM', 'logfat_risk3_lag2']


Let's now run all the experiments and save the results:

In [49]:
import os
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

def run_and_save_experiment(experiment_name, file_prefix, feature_list, df, model=None, results_folder="results"):
    print("\n" + "="*60)
    print(f"RUNNING EXPERIMENT: {experiment_name}")
    print(f"Features count: {len(feature_list)}")
    print("="*60)
    
    os.makedirs(results_folder, exist_ok=True)
    
    # 1. Prepare data (Ahora retiene los NaNs de las crisis continuas para usarlos en el bucle)
    X, y = prepare_ml_experiment(df, features_to_use=feature_list, keep_target_nans=True)
    periods = X.index.get_level_values('month')

    # 2. Split strategy
    panel_split = PanelSplit(periods=periods, n_splits=24, test_size=1, gap=1)

    # 3. Initialize Model
    clf = model if model is not None else RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1
    )

    all_y_true, all_y_prob, all_y_pred = [], [], []
    all_iso3, all_months = [], []

    # 4. Cross-Validation Loop
    for train_idx, test_idx in panel_split.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        # --- APLICACIÓN DE TU REGLA MAESTRA ---
        
        # PARA EL TRAIN: Borramos los NaNs para que el modelo solo aprenda de onsets puros
        train_mask = y_train.notna()
        X_train_clean = X_train[train_mask]
        y_train_clean = y_train[train_mask]
        
        # PARA EL TEST: Los NaNs (crisis) se vuelven 0. Si el modelo predice 1, será un Falso Positivo.
        y_test_clean = y_test.fillna(0)
        
        # ----------------------------------------
        
        # Entrenamos con los datos limpios
        clf.fit(X_train_clean, y_train_clean)
        
        # Predecimos sobre el test (X_test no tiene NaNs en las features)
        probs = clf.predict_proba(X_test)[:, 1]
        preds = clf.predict(X_test)
        
        # Guardamos el target del test ya rellenado con 0 para evaluar el rendimiento real
        all_y_true.extend(y_test_clean.values)
        all_y_prob.extend(probs)
        all_y_pred.extend(preds)

        all_iso3.extend(X_test.index.get_level_values('iso3'))
        all_months.extend(X_test.index.get_level_values('month'))

    # 5. SAVE FILE 1: Predictions
    df_preds = pd.DataFrame({
        'iso3': all_iso3,
        'month': all_months,
        'y_true': all_y_true,
        'y_prob': all_y_prob
    })
    pred_path = os.path.join(results_folder, f"{file_prefix}_predictions.csv")
    df_preds.to_csv(pred_path, index=False)
    print(f"Saved predictions to: {pred_path}")
    
    # 6. SAVE FILE 2: Feature Importance 
    if hasattr(clf, "feature_importances_"):
        df_feat = pd.DataFrame({
            'feature': X.columns,
            'importance': clf.feature_importances_
        }).sort_values(by='importance', ascending=False)
        feat_path = os.path.join(results_folder, f"{file_prefix}_feature_importance.csv")
        df_feat.to_csv(feat_path, index=False)
        print(f"Saved feature importance to: {feat_path}")
    else:
        print("Skipped feature importance (Este modelo no utiliza features)")
    
    # 7. Quick report
    print(f"F1: {f1_score(all_y_true, all_y_pred):.3f} | AUC: {roc_auc_score(all_y_true, all_y_prob):.3f}")

Our first experiment is going to be our benchmark model. This id going to be a random classifier respecting the proportions of the clases.

In [50]:
run_and_save_experiment(
    experiment_name="Random Benchmark (Stratified)", 
    file_prefix="benchmark_random", 
    feature_list=[], 
    df=df,
    model=DummyClassifier(strategy='stratified', random_state=42) 
)


RUNNING EXPERIMENT: Random Benchmark (Stratified)
Features count: 0
Saved predictions to: results/benchmark_random_predictions.csv
Skipped feature importance (Este modelo no utiliza features)
F1: 0.000 | AUC: 0.500


In [51]:
run_and_save_experiment(
    experiment_name="Optimal", 
    file_prefix="optimal", 
    feature_list=best_general, 
    df=df, 
)


RUNNING EXPERIMENT: Optimal
Features count: 14
Saved predictions to: results/optimal_predictions.csv
Saved feature importance to: results/optimal_feature_importance.csv
F1: 0.189 | AUC: 0.901


In [102]:
run_and_save_experiment(
    experiment_name="Best", 
    file_prefix="best", 
    feature_list= best_features_derived_humanitarian_impact + ["INFORM"] + ['risk_3'] + ['monthly_displacement_lag1','monthly_displacement_lag2','disp_diff_3m_6m','fat_diff_3m_6m','bat_gr90','exp_gr150'] + features_cerf_signals,
    df=df, 
)


RUNNING EXPERIMENT: Best
Features count: 30
Saved predictions to: results/best_predictions.csv
Saved feature importance to: results/best_feature_importance.csv
F1: 0.189 | AUC: 0.890


Now let's try training the model with all the possible combination of features

In [53]:
# DEFINE DISCRETE OPTION STATES FOR EACH FEATURE BLOCK
feature_dimensions = {
    "Baseline": [
        None, 
        {"label": "Baseline", "suffix": "baseline", "features": features_baseline}
    ],
    "Derived Impact": [
        None, 
        {"label": "Derived Humanitarian Impact", "suffix": "derived_humanitarian_impact", "features": features_derived_humanitarian_impact},
        {"label": "Derived Humanitarian Impact BEST", "suffix": "derived_humanitarian_impact_best", "features": best_features_derived_humanitarian_impact}
    ],
    "Risk Alerts": [
        None, 
        {"label": "Risk Alerts", "suffix": "risk_alerts", "features": features_risk_alerts},
        {"label": "Risk Alerts BEST", "suffix": "risk_alerts_best", "features": best_features_risk_alerts}
    ],
    "INFORM": [
        None, 
        {"label": "INFORM", "suffix": "inform", "features": features_inform}
    ],
    "Signals (CERF)": [
        None, 
        {"label": "Signals (CERF)", "suffix": "signals_cerf", "features": features_cerf_signals}
    ]
}


# COMPUTE THE CARTESIAN PRODUCT & BUILD CONFIGURATIONS
experiment_configs = []
experiments_map = {}

keys = list(feature_dimensions.keys())
all_dimension_options = [feature_dimensions[k] for k in keys]

for combo in itertools.product(*all_dimension_options):
    active_blocks = [block for block in combo if block is not None]
    
    if not active_blocks:
        continue
        
    block_names = [b["label"] for b in active_blocks]
    block_suffixes = [b["suffix"] for b in active_blocks]
    
    if len(active_blocks) == 5:
        is_pure_general = not any("BEST" in b["label"] for b in active_blocks)
        
        has_no_raw_derived = "Derived Humanitarian Impact" not in block_names
        has_no_raw_risk = "Only Risk Alerts" not in block_names
        has_no_raw_signals = "Only Signals (CERF)" not in block_names
        is_all_best = has_no_raw_derived and has_no_raw_risk and has_no_raw_signals
        
        if is_all_best:
            name = "ALL FEATURES (With BEST Options)"
            prefix = "all_features_best"
        elif is_pure_general:
            name = "ALL FEATURES (Baseline + Derived Impact + Risk Alerts + INFORM + Signals CERF)"
            prefix = "all_features"
        else:
            name = f"ALL FEATURES (Hybrid Mix: {' + '.join(block_names)})"
            prefix = f"all_features_hybrid_{'_'.join(block_suffixes)}"
    else:
        name = " + ".join(block_names)
        prefix = "_".join(block_suffixes)
        
    # Save in the dynamic map
    experiments_map[name] = prefix

    combined_features = []
    for b in active_blocks:
        for feature_name in b["features"]:
            if feature_name not in combined_features:
                combined_features.append(feature_name)
                
    experiment_configs.append({
        "name": name,
        "prefix": prefix,
        "features": combined_features
    })

# EXECUTE THE AUTOMATED SEAMLESS TRAINING GRID LOOP
print(f"Successfully generated {len(experiment_configs)} valid experimental configurations.")
print("CRITICAL GUARDRAIL: No experiment will mix a general feature block with its own BEST variant.\n")

for config in experiment_configs:
    run_and_save_experiment(
        experiment_name=config["name"],
        file_prefix=config["prefix"],
        feature_list=config["features"],
        df=df 
    )

Successfully generated 71 valid experimental configurations.
CRITICAL GUARDRAIL: No experiment will mix a general feature block with its own BEST variant.


RUNNING EXPERIMENT: Signals (CERF)
Features count: 7
Saved predictions to: results/signals_cerf_predictions.csv
Saved feature importance to: results/signals_cerf_feature_importance.csv
F1: 0.129 | AUC: 0.795

RUNNING EXPERIMENT: INFORM
Features count: 6
Saved predictions to: results/inform_predictions.csv
Saved feature importance to: results/inform_feature_importance.csv
F1: 0.168 | AUC: 0.874

RUNNING EXPERIMENT: INFORM + Signals (CERF)
Features count: 13
Saved predictions to: results/inform_signals_cerf_predictions.csv
Saved feature importance to: results/inform_signals_cerf_feature_importance.csv
F1: 0.155 | AUC: 0.876

RUNNING EXPERIMENT: Risk Alerts
Features count: 37
Saved predictions to: results/risk_alerts_predictions.csv
Saved feature importance to: results/risk_alerts_feature_importance.csv
F1: 0.123 | AUC: 0.884

RUNNING

In [54]:
import json
route_save = "results/experiments_map.json" 

with open(route_save, "w", encoding="utf-8") as f:    json.dump(experiments_map, f, indent=4, ensure_ascii=False)

print(f"Map of experiments saved successfully in: {route_save}")

Map of experiments saved successfully in: results/experiments_map.json
